In [2]:
import os
import sys
from pathlib import Path
from openai import OpenAI

# playground 로 옮긴 뒤에도 키가 잡히게 (.env / env 직접 읽음, dotenv 없음)
HERE = Path.cwd()
if HERE.name != "llm-api-playground" and (HERE / "llm-api-playground").is_dir():
    HERE = HERE / "llm-api-playground"
for env_path in (HERE / ".env", HERE / "env"):
    if env_path.exists():
        for line in env_path.read_text(encoding="utf-8").splitlines():
            if "=" in line and "key" in line.lower():
                os.environ["OPENAI_API_KEY"] = line.split("=", 1)[1].strip()

print("cwd:", HERE)
print("python:", sys.executable)
API_MODEL = "gpt-5.4-nano"
client = OpenAI()
print("키 로드:", "OK" if os.environ.get("OPENAI_API_KEY") else "실패 — env/.env 확인")


In [3]:
client = OpenAI()


In [ ]:
r = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": f"이 리뷰 분석해. {review}"}],
    max_completion_tokens=300,
)
print(r.choices[0].message.content)

In [10]:
review = "배송은 느렸지만 물건은 기대 이상이야. 또 사줄게!" # 쇼핑몰 리뷰
r = client.chat.completions.create(
    model=API_MODEL,
    messages=[{
        "role": "user",
        "content": f"이 리뷰를 JSON으로 분석해. {review}"
    }],
    response_format={"type": "json_object"},
    max_completion_tokens=300,
)
print(r.choices[0].message.content)

{
  "language": "ko",
  "original_review": "배송은 느렸지만 물건은 기대 이상이야. 또 사줄게!",
  "sentiment": {
    "overall": "positive",
    "score_estimate": 0.55,
    "reasoning": [
      "배송 지연 불만이 있음(부정 요소)",
      "제품 품질 만족: '기대 이상'",
      "재구매 의사 표현: '또 사줄게!'"
    ]
  },
  "aspects": [
    {
      "aspect": "배송",
      "polarity": "negative",
      "evidence": "배송은 느렸지만"
    },
    {
      "aspect": "제품/품질",
      "polarity": "positive",
      "evidence": "물건은 기대 이상이야"
    },
    {
      "aspect": "재구매 의사",
      "polarity": "positive",
      "evidence": "또 사줄게!"
    }
  ],
  "summary": "배송은 느렸지만 제품 만족도가 높고 재구매 의사가 있는 긍정 리뷰."
}


In [14]:
import json

print(type(r.choices[0].message.content))
data = json.loads(r.choices[0].message.content)
print(data)
print(type(data))
print(data.keys())

<class 'str'>
{'language': 'ko', 'raw_review': '배송은 느렸지만 물건은 기대 이상이야. 또 사줄게!', 'sentiment': {'overall': '긍정', 'score_hint': 0.6, 'reason': ['배송 지연(부정 요소)이 있으나', '물건의 품질/만족도가 기대 이상(긍정 요소)', "추가 구매 의사 표현('또 사줄게!')"]}, 'aspects': [{'aspect': '배송', 'polarity': '부정', 'evidence': '배송은 느렸지만'}, {'aspect': '상품 품질/만족도', 'polarity': '긍정', 'evidence': '물건은 기대 이상이야'}, {'aspect': '재구매 의사', 'polarity': '긍정', 'evidence': '또 사줄게!'}], 'summary': '배송은 느렸지만 상품은 기대 이상이었고 재구매 의사도 있음.'}
<class 'dict'>
dict_keys(['language', 'raw_review', 'sentiment', 'aspects', 'summary'])


In [13]:
for i in range(3):
    r = client.chat.completions.create(
    model=API_MODEL,
    messages=[{
        "role": "user",
        "content": f"이 리뷰를 JSON으로 분석해. {review}"
    }],
    response_format={"type": "json_object"},
    max_completion_tokens=300,
)
print(r.choices[0].message.content)

{
  "language": "ko",
  "raw_review": "배송은 느렸지만 물건은 기대 이상이야. 또 사줄게!",
  "sentiment": {
    "overall": "긍정",
    "score_hint": 0.6,
    "reason": [
      "배송 지연(부정 요소)이 있으나",
      "물건의 품질/만족도가 기대 이상(긍정 요소)",
      "추가 구매 의사 표현('또 사줄게!')"
    ]
  },
  "aspects": [
    {
      "aspect": "배송",
      "polarity": "부정",
      "evidence": "배송은 느렸지만"
    },
    {
      "aspect": "상품 품질/만족도",
      "polarity": "긍정",
      "evidence": "물건은 기대 이상이야"
    },
    {
      "aspect": "재구매 의사",
      "polarity": "긍정",
      "evidence": "또 사줄게!"
    }
  ],
  "summary": "배송은 느렸지만 상품은 기대 이상이었고 재구매 의사도 있음."
}


In [16]:
#응답 재현 반복
# JSON이 아닌 텍스트가 섞이는 경우 400 발생
# 모두 JSON 읃답으로 받더라도, 매번 스키마가 변경되어 원하는 모양으로 받을 수 없다.
for i in range(3):
    r = client.chat.completions.create(
        model=API_MODEL,
        messages=[{
            "role": "user",
            "content": f"이 리뷰를 JSON으로 분석해. {review}"
        }],
        response_format={"type": "json_object"},
        max_completion_tokens=300,
    )
    data = json.loads(r.choices[0].message.content)
    print(data.keys())

dict_keys(['language', 'review_original', 'sentiment_overall', 'topics', 'intent_signals', 'structured_summary'])
dict_keys(['language', 'review_summary', 'sentiment_by_aspect', 'key_phrases', 'repurchase_intent_detected', 'repurchase_intent_evidence'])
dict_keys(['language', 'original_review', 'summary', 'aspects', 'review_rewrite'])


In [19]:
from openai import BadRequestError
import json

review = "배송은 느렸지만 물건은 기대 이상이야. 또 사줄게!" # 쇼핑몰 리뷰
try: 
    #API 요청 http-post 요청
    r = client.chat.completions.create(
        model=API_MODEL,
        messages=[{
            "role": "user",
            "content": f"이 리뷰를 JSON으로 분석해. {review}"
        }],
        response_format={"type": "json_object"},
        max_completion_tokens=300,
    )
    data = json.loads(r.choices[0].message.content)
    print(data.keys())

except BadRequestError as e:
    print(e.status_code, str(e))
    #print(r.choices[0].message.content)
    #response_format={"typr":"json_object"} 이렇게 설정했을 때는 반드시 "JSON"이라는 단어가 들어가야 한다.


dict_keys(['sentiment', 'aspects', 'summary', 'entities', 'language'])


In [21]:
r = client.chat.completions.create(
    model = API_MODEL,
    messages=[{"role":"user", "content": f"이 리뷰를 JSON으로 분석해줘. {review}"}],
    response_format={"type": "json_schema"},
    max_completion_tokens=300
)
data=json.loads(r.choices[0].message.content)
print(data.keys())                     

BadRequestError: Error code: 400 - {'error': {'message': "Missing required parameter: 'response_format.json_schema'.", 'type': 'invalid_request_error', 'param': 'response_format.json_schema', 'code': 'missing_required_parameter'}}

In [ ]:
from openai import BadRequestError
schema = {
    "type": "object",
    "properties": {
        "감정": {"type": "string", "description": "긍정/부정/중립 중에서 하나 선택"},
        "별점": {"type": "integer", "description": "1~5점 사이 점수, 높은 게 긍정"},
        "요약": {"type": "string", "description": "한 문장으로 요약"},
    },
    "additionalProperties": False,
    "required": ["감정", "별점", "요약"]
}

try:
    r = client.chat.completions.create(
        model = API_MODEL,
        messages=[{"role":"user", "content": f"이 리뷰를 JSON으로 분석해줘. {review}"}],
        response_format={"type": "json_schema",
                         "json_schema": {"name":"review_analysis", "strict":True, "schema":schema}},
        max_completion_tokens=300,
    )

    data = json.loads(r.choices[0].message.content)
    print(data)
except BadRequestError as e:
    print(e)



{'감정': '긍정', '별점': 4, '요약': '배송은 느렸지만 제품 만족도가 높아 기대 이상이며, 재구매 의사도 있어 전반적으로 긍정적인 리뷰입니다.'}


In [ ]:
# 엄격한 코드 검증 Schema
from openai import BadRequestError
import json

schema = {
    "type": "object",
    "properties": {
        "감정": {"type": "string", "description": "긍정/부정/중립 중에서 하나 선택"},
        "별점": {"type": "integer", "description": "1~5점 사이 점수, 높은 게 긍정"},
        "요약": {"type": "string", "description": "한 문장으로 요약"},
    },
    "required": ["감정", "별점", "요약"],
    "additionalProperties": False,
}

try:
    r = client.chat.completions.create(
        model=API_MODEL,
        messages=[{"role": "user", "content": f"이 리뷰를 JSON으로 분석해줘. {review}"}],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "review_analysis",
                "strict": True,
                "schema": schema,
            },
        },
        max_completion_tokens=300,
    )
    data = json.loads(r.choices[0].message.content)
    print(data)
except BadRequestError as e:
    print(e)


{'감정': 'good', '별점': 4, '요약': '배송은 느렸지만 제품 만족도가 기대 이상이라 재구매 의사가 있어 전반적으로 긍정적인 리뷰입니다.'}


In [ ]:
# Pydantic 클래스
from pydantic import BaseModel, Field
from openai import BadRequestError

review = "배송은 느렸지만 물건은 기대 이상이야. 또 사줄게!"

class ReviewAnalysis(BaseModel):
    감정: str = Field(description="긍정/부정/중립 중에서 하나 선택")
    별점: int = Field(description="1~5점 사이 점수, 높은 게 긍정")
    요약: str = Field(description="한 문장으로 요약")

try:
    r = client.chat.completions.parse(
        model=API_MODEL,
        messages=[{
            "role": "user",
            "content": f"이 리뷰를 JSON으로 분석해줘. {review}",
        }],
        response_format=ReviewAnalysis,
        max_completion_tokens=300,
    )

    data = r.choices[0].message.parsed
    print(data)
    print(data.감정, data.별점, data.요약)

except BadRequestError as e:
    print(e)